This notebook provides an example of loading and summarizing results using `dev.lib.results`.
Copy it and adjust as needed.

# Imports

In [1]:
import functools
from glob import glob
from typing import Any

import pandas as pd
from loguru import logger

import dev.lib.data as devdata
import dev.lib.results as devresults
import lib.util

lib.util.init_notebook(torch_=False)
pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 0)

# Quick example

**Note:** see the docstring of `dev.lib.results.load_dataframe` for a complete reference.

In [2]:
# Example: load and aggregate results of multiple experiments.
devresults.load_dataframe(
    glob('exp/examples/mlp/0/*/evaluation'),  # Name='A'
    glob('exp/examples/mlp/0/*/ensemble-5'),  # Name='B',
).groupby(['Dataset.name', 'Name'])['metrics.test.score'].agg(['mean', 'std']).round(4)

mean     std
Dataset.name Name                
california   A    -0.4942  0.0051
             B    -0.4874  0.0029
churn        A     0.8577  0.0019
             B     0.8595  0.0020

# Main example

## Presentation

In [3]:
def show_column(series: pd.Series) -> None:
    assert isinstance(series.name, str)
    try:
        prefix, suffix = series.name.rsplit('.', 1)
    except ValueError:
        raise ValueError(f'Cannot summarize the column: {series.name}')

    if suffix == 'rank':
        print(f'Ranks by {prefix}')
        display(series.groupby('Name').agg(['mean', 'std']).sort_values('mean'))

    elif prefix.endswith('.relative'):
        print(f'Relative to {suffix}')
        display(
            series.groupby('Name')
            .agg(
                **{
                    'mean': 'mean',
                    'std': 'std',
                    'iqm1': lambda x: devresults.interquantile_mean(x, None, 0.25),
                    'iqm23': lambda x: devresults.interquantile_mean(x, 0.25, 0.75),
                    'iqm4': lambda x: devresults.interquantile_mean(x, 0.75, None),
                    'min': 'min',
                    # 'q10': lambda x: x.quantile(0.1),
                    # 'q25': lambda x: x.quantile(0.25),
                    'median': 'median',
                    # 'q75': lambda x: x.quantile(0.75),
                    # 'q90': lambda x: x.quantile(0.9),
                    'max': 'max',
                }
            )
            .sort_values('mean', ascending=False)
        )

    else:
        raise ValueError(f'Cannot summarize the column: {series.name}')


@functools.lru_cache(None)
def _format_column_name(c: str) -> str:
    if c.startswith('metrics.'):
        c = c.removeprefix('metrics.')
        c = c.replace('.score.', '.')
        c = c.replace('.unified_score.', '.unified.')
        c = c.removesuffix('.mean').replace('.mean.', '.')

        if c.endswith('.std'):
            c = f'{c.removesuffix(".std")}~'
        elif c.endswith('.rank'):
            c = f'{c.removesuffix(".rank")}^'
        elif c.rsplit('.', 1)[0].endswith('.relative'):
            c = c.replace('.unified.', '.')
            c = c.replace('.relative.', ':')

    elif c.startswith('config.'):
        # NOTE: collisions are possible
        c = c.rsplit('.', 1)[-1]

    return c


def format_dataframe(df: pd.DataFrame, *, precision: int = 4) -> pd.DataFrame:
    """Format the dataframe for the final presentation."""
    df = df.copy()

    # Format dataset names.
    df.index = df.index.set_levels(  # type: ignore
        [
            x.rsplit('data/', 1)[-1].capitalize()
            for x in df.index.levels[df.index.names.index('Dataset.name')]  # type: ignore
        ],
        level='Dataset.name',
    )

    # Format metrics.
    score_columns = [x for x in df.columns if x.endswith('.score.mean')]
    df[score_columns] = df[score_columns].abs()
    df.loc[
        df.index.get_level_values('Dataset.name') == 'House',
        [x for x in df.columns if x.endswith(('.score.mean', '.score.std'))],
    ] /= 10000
    df = df.round(precision)

    # Format index.
    df.index.names = [{'Dataset.name': 'Dataset'}.get(x, x) for x in df.index.names]

    # Format columns.
    column_mapping = {}
    for column in df.columns:
        formatted_column = _format_column_name(column)
        if formatted_column != column:
            column_mapping[column] = formatted_column
    assert len(set(column_mapping.values())) == len(column_mapping), (
        'Formatted column names contain duplicates'
    )
    if column_mapping:
        df.rename(columns=column_mapping, inplace=True)

    return df


def show_dataframe(df: pd.DataFrame) -> None:
    print('Main')
    display(format_dataframe(df))

## Computation

In [4]:
def build_dataframes(
    records: list[dict[str, Any]],
    num_statistics: None | list[str] = None,
    reference_names: None | list[str] = None,
    absolute_metrics: None | list[str] = None,
    drop_reference_rows: bool = False,
    rank_columns: None | list[str] = None,
    sort_by: None | list[str] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Args:
        records: the records loaded with `devresults.load_records`.
        num_statistics: the statistics for `devresults.aggregate_all_columns`.
        reference_names: the reference names for `devresults.compute_relative_metrics_`.
        absolute_metrics: the metric columns for `devresults.compute_relative_metrics_`.
        drop_reference_rows: if True, the rows with the reference names will be dropped.
        rank_columns: the columns for rank computation.
        sort_by: the user-defined part of the sorting columns.
    """
    # Create the dataframe.
    df = pd.DataFrame.from_records(records)
    first_name = df[['Name']].iloc[0, 0]

    # Aggregate all columns.
    df = devresults.aggregate_all_columns(
        df,
        ['mean', 'std'] if num_statistics is None else num_statistics,
        by=['Dataset.name', 'Name'],
    )

    # Compute relative metrics.
    for reference_name in [first_name] if reference_names is None else reference_names:
        devresults.compute_relative_metrics_(
            df,
            reference_name,  # type: ignore
            (
                ['metrics.test.unified_score.mean']
                if absolute_metrics is None
                else absolute_metrics
            ),
            by='Dataset.name',
        )
        if drop_reference_rows:
            df = df.loc[df.index.get_level_values('Name') != reference_name]

    # Drop datasets where not all models are available.
    # This must be done if you plan to compute ranks and/or summarize columns.
    # Otherwise, results can be misleading.
    complete_df = devresults.drop_incomplete_datasets(df)
    all_datasets_are_complete = len(df) == len(complete_df)
    if not all_datasets_are_complete:
        incomplete_datasets = set(df.index.unique('Dataset.name')) - set(
            complete_df.index.unique('Dataset.name')
        )
        logger.warning(f'Incomplete datasets: {", ".join(incomplete_datasets)}')

    # Compute ranks.
    default_rank_column = 'metrics.test.score'
    for rank_column in (
        ([default_rank_column] if f'{default_rank_column}.mean' in df.columns else [])
        if rank_columns is None
        else rank_columns
    ):
        df = devresults.compute_ranks(
            df, f'{rank_column}.mean', f'{rank_column}.std', by='Dataset.name'
        )
        complete_df = (
            df
            if all_datasets_are_complete
            else devresults.compute_ranks(
                complete_df,
                f'{rank_column}.mean',
                f'{rank_column}.std',
                by='Dataset.name',
            )
        )

    # Sort the dataframes.
    for df_ in (df, complete_df):
        df_.sort_values(
            [
                'Dataset.train_size',
                'Dataset.name',
                *(['metrics.test.score.mean'] if sort_by is None else sort_by),
                'Name',
            ],
            ascending=[True, True, False, True],
            inplace=True,
        )

    return df, complete_df

In [ ]:
# NOTE
# Clear the cache when experiment files change on disk.
# devresults.clear_cache()

datasets = [
    devdata.CHURN,
    devdata.CALIFORNIA,
    # *devdata.INDEPENDENT_DATASETS,
    # *devdata.TABRED_DATASETS,
    # *devdata.TABARENA_DATASETS,
]

records = []
records += devresults.load_records(
    {
        # Layout: exp/{{ model }}/{{ variant }}/{{ dataset }}/{{ activity }}
        'MLP': [f'exp/examples/mlp/0/{d}/evaluation' for d in datasets],
        'MLP(E5)': [f'exp/examples/mlp/0/{d}/ensemble-5' for d in datasets],
    },
    allow_skipping=True,
    cache=True,
)
# Use `devresults.load_records` multiple times if needed. This can be relevant
# when expressing all necessary `load_fn` functions at once is not possible.
# records += devresults.load_records(other_experiments, custom_load_fn)
df, complete_df = build_dataframes(records)

# Note that `complete_df` is used for summarizing columns.
print()
show_column(complete_df['metrics.test.score.mean.rank'])
show_column(complete_df['metrics.test.unified_score.mean.relative.MLP'])
show_dataframe(
    df[
        [
            # Metrics
            'metrics.test.score.mean.rank',
            'metrics.test.score.mean',
            'metrics.test.score.std',
            'metrics.val.score.mean',
            'metrics.val.score.std',
            'metrics.train.score.mean',
            'metrics.train.score.std',
            # Hyperparameters
            'config.optimizer.lr',
        ]
    ]
)


Ranks by metrics.test.score.mean


,mean,std
Name,,
MLP(E5),1.0,0.000000
MLP,1.5,0.707107


Relative to MLP


,mean,std,iqm1,iqm23,iqm4,min,median,max
Name,,,,,,,,
MLP(E5),0.421658,0.294014,0.213758,NaN,0.629557,0.213758,0.421658,0.629557
MLP,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000


Main


test^    test   test~     val    val~   train  train~  \
Dataset    Name                                                             
Churn      MLP(E5)      1  0.8595  0.0020  0.8608  0.0028  0.8654  0.0005   
           MLP          1  0.8577  0.0019  0.8637  0.0026  0.8653  0.0027   
California MLP(E5)      1  0.4874  0.0029  0.4836  0.0013  0.3785  0.0094   
           MLP          2  0.4942  0.0051  0.4901  0.0037  0.3868  0.0225   

                        lr  
Dataset    Name             
Churn      MLP(E5)     NaN  
           MLP      0.0006  
California MLP(E5)     NaN  
           MLP      0.0009